In [1]:

from pathlib import Path
import os
import sys
import json
import hashlib
import platform

import numpy as np
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("=== VIREÁX Quantum Compilation: Environment Audit ===")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Input directory exists:", INPUT_ROOT.exists())
print("Working directory:", WORK_ROOT)

print("\n=== Mounted Input Files ===")
if INPUT_ROOT.exists():
    files = sorted(p for p in INPUT_ROOT.rglob("*") if p.is_file())
    if not files:
        print("No files found. Attach the competition dataset or starter kit.")
    for path in files:
        print(f"{path} ({path.stat().st_size:,} bytes)")
else:
    print("Kaggle input directory is unavailable.")

print("\n=== Relevant Packages ===")
for package in ["kagglehub", "pytest", "qiskit", "cirq"]:
    try:
        mod = __import__(package)
        print(f"{package}: available",
              getattr(mod, "__version__", "version unknown"))
    except ImportError:
        print(f"{package}: not installed")

print("\n=== Expected Competition Assets ===")
all_paths = [str(p).lower() for p in INPUT_ROOT.rglob("*")] if INPUT_ROOT.exists() else []

expected = [
    "verifiable_circuit_compilation.pdf",
    "scoring_readme.md",
    "invsqrt.kmb",
    "dqi.kmb",
    "submission.npz",
    "kickmix",
    "verifiers",
]

for item in expected:
    matches = [p for p in all_paths if item in p]
    print(f"{item}: {'FOUND' if matches else 'not detected'}")
    for match in matches[:5]:
        print("  ", match)

print("\nAudit complete. No circuit was modified or submitted.")


=== VIREÁX Quantum Compilation: Environment Audit ===
Python: 3.13.15
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
NumPy: 2.1.3
Pandas: 2.3.3
Input directory exists: True
Working directory: /kaggle/working

=== Mounted Input Files ===
/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/kickmix/LICENSE.txt (11,358 bytes)
/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/kickmix/kickmix.so (3,220,392 bytes)
/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission/dfthcsos/angle_k_0.kmb (1,671,932 bytes)
/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission/dfthcsos/angle_k_1.kmb (1,662,896 bytes)
/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission/dfthcsos/angle_k_2.kmb (1,665,821 bytes)
/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission/dfthcsos/angle_k_3.kmb (1,666,802 bytes)
/kaggle/input/com

In [2]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/competitions/"
    "google-verifiable-quantum-circuit-compilation"
)

for task in ["invsqrt", "dqi", "dfthcsos"]:
    print(f"\n=== {task} verifier ===")
    folder = ROOT / "verifiers" / task
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            print(path.relative_to(ROOT), path.stat().st_size, "bytes")

print("\n=== dfthcsos scoring guide ===")
readme = ROOT / "verifiers" / "dfthcsos" / "scoring_readme.md"
if readme.exists():
    print(readme.read_text(errors="replace"))

print("\n=== Kickmix runtime ===")
for path in sorted((ROOT / "kickmix").rglob("*")):
    if path.is_file():
        print(path.relative_to(ROOT))


=== invsqrt verifier ===
verifiers/invsqrt/__init__.py 84 bytes
verifiers/invsqrt/nr_reference_circuit.py 18862 bytes
verifiers/invsqrt/verify_submission_invsqrt.py 17210 bytes

=== dqi verifier ===
verifiers/dqi/__init__.py 87 bytes
verifiers/dqi/dqi_reference_circuit.py 30450 bytes
verifiers/dqi/verify_submission_dqi.py 20959 bytes

=== dfthcsos verifier ===
verifiers/dfthcsos/FeMoCo76o113e.fcidump 128128348 bytes
verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz 61028 bytes
verifiers/dfthcsos/SCORING_README.md 22477 bytes
verifiers/dfthcsos/__init__.py 326 bytes
verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py 10258 bytes
verifiers/dfthcsos/scoreinfra/__init__.py 280 bytes
verifiers/dfthcsos/scoreinfra/circuits.py 24418 bytes
verifiers/dfthcsos/scoreinfra/costs_prepare.py 8335 bytes
verifiers/dfthcsos/scoreinfra/dfthc_math.py 17608 bytes
verifiers/dfthcsos/scoreinfra/reference.py 13180 bytes
verifiers/dfthcsos/verify.py 31752 bytes

=== dfthcsos scoring guide ===


In [3]:

from pathlib import Path
import sys
import json
import hashlib
import traceback
from datetime import datetime, timezone

ROOT = Path(
    "/kaggle/input/competitions/"
    "google-verifiable-quantum-circuit-compilation"
)
VERIFIERS = ROOT / "verifiers"
KICKMIX = ROOT / "kickmix"
BASELINE = ROOT / "sample_submission" / "dfthcsos"

OUT = Path("/kaggle/working/vireax_runs")
OUT.mkdir(parents=True, exist_ok=True)
REPORT = OUT / "dfthcsos_baseline_verification.json"

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Validate all expected baseline artifacts before importing the verifier.
expected_angles = [
    BASELINE / f"angle_k_{k}.kmb"
    for k in range(10)
]
required = [
    VERIFIERS / "dfthcsos" / "verify.py",
    BASELINE / "submission.npz",
    KICKMIX / "kickmix.so",
    *expected_angles,
]

missing = [str(p) for p in required if not p.is_file()]

report = {
    "task": "dfthcsos",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "baseline_directory": str(BASELINE),
    "verifier": str(VERIFIERS / "dfthcsos" / "verify.py"),
    "status": "NOT_RUN",
    "input_files": {},
}

if missing:
    report["status"] = "ERROR"
    report["error_type"] = "FileNotFoundError"
    report["error"] = "Required competition files missing."
    report["missing_files"] = missing
else:
    # Fingerprint every required artifact, without modifying it.
    for path in required:
        if path.is_file():
            report["input_files"][str(path.relative_to(ROOT))] = {
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }

    for path in [str(KICKMIX), str(VERIFIERS)]:
        if path not in sys.path:
            sys.path.insert(0, path)

    try:
        import numpy
        import scipy
        import kickmix
        from dfthcsos.verify import verify

        report["dependencies"] = {
            "numpy": numpy.__version__,
            "scipy": scipy.__version__,
            "kickmix_import": "SUCCESS",
        }

        print("Running official DFTHCSOS baseline verifier...")
        print("Competition input files will not be modified.")

        result = verify(str(BASELINE))

        if isinstance(result, dict):
            report["result"] = result
            status = result.get("status")
            report["status"] = (
                str(status).upper()
                if status is not None
                else "RETURNED_WITHOUT_STATUS"
            )
        else:
            report["result_type"] = type(result).__name__
            report["result_repr"] = repr(result)
            report["status"] = "RETURNED_UNEXPECTED_RESULT"

    except Exception as exc:
        report["status"] = "ERROR"
        report["error_type"] = type(exc).__name__
        report["error"] = str(exc)
        report["traceback"] = traceback.format_exc()

REPORT.write_text(json.dumps(report, indent=2, default=str))

print("\n=== VIREÁX BASELINE AUDIT ===")
print(json.dumps(report, indent=2, default=str))
print("\nReport saved to:", REPORT)

if report["status"] != "PASS":
    print(
        "\nBaseline PASS is not confirmed. "
        "Do not begin optimisation until the result is understood."
    )
else:
    print(
        "\nBaseline verifier returned PASS. "
        "This establishes the verifier's reported result, "
        "not a proof of global optimality."
    )


Running official DFTHCSOS baseline verifier...
Competition input files will not be modified.

=== VIREÁX BASELINE AUDIT ===
{
  "task": "dfthcsos",
  "timestamp_utc": "2026-10-26T06:07:47.227685+00:00",
  "baseline_directory": "/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission/dfthcsos",
  "verifier": "/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/verifiers/dfthcsos/verify.py",
  "status": "PASS",
  "input_files": {
    "verifiers/dfthcsos/verify.py": {
      "bytes": 31752,
      "sha256": "0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d"
    },
    "sample_submission/dfthcsos/submission.npz": {
      "bytes": 4537436,
      "sha256": "14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69"
    },
    "kickmix/kickmix.so": {
      "bytes": 3220392,
      "sha256": "c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e"
    },
    "sample_submission/dfthcsos/angle_k_0.kmb": {
     

In [4]:

from pathlib import Path
import sys
import json
import hashlib
import traceback
from datetime import datetime, timezone

ROOT = Path(
    "/kaggle/input/competitions/"
    "google-verifiable-quantum-circuit-compilation"
)
VERIFIERS = ROOT / "verifiers"
KICKMIX = ROOT / "kickmix"
BASELINE = ROOT / "sample_submission" / "dfthcsos"

OUT = Path("/kaggle/working/vireax_runs")
OUT.mkdir(parents=True, exist_ok=True)

REPORT = OUT / "dfthcsos_baseline_verification.json"

# Validate the actual paths before importing anything.
required = [
    VERIFIERS / "dfthcsos" / "verify.py",
    BASELINE / "submission.npz",
    KICKMIX / "kickmix.so",
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required competition files missing:\n" + "\n".join(missing)
    )

# Make the competition's own modules importable.
for path in [str(KICKMIX), str(VERIFIERS)]:
    if path not in sys.path:
        sys.path.insert(0, path)

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

report = {
    "task": "dfthcsos",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "baseline_directory": str(BASELINE),
    "baseline_npz_sha256": sha256(BASELINE / "submission.npz"),
    "verifier": str(VERIFIERS / "dfthcsos" / "verify.py"),
    "status": "NOT_RUN",
}

try:
    import numpy
    import scipy
    import kickmix
    from dfthcsos.verify import verify

    report["dependencies"] = {
        "numpy": numpy.__version__,
        "scipy": scipy.__version__,
        "kickmix_import": "SUCCESS",
    }

    print("Running official dfthcsos baseline verifier...")
    print("Original input files will not be modified.")
    print("This may take approximately 25 seconds, according to the guide.")

    result = verify(str(BASELINE))

    # Convert the result to ordinary JSON-compatible data.
    report["status"] = result.get("status", "RETURNED_WITHOUT_STATUS")
    report["result"] = result

except Exception as exc:
    report["status"] = "ERROR"
    report["error_type"] = type(exc).__name__
    report["error"] = str(exc)
    report["traceback"] = traceback.format_exc()

REPORT.write_text(json.dumps(report, indent=2, default=str))

print("\n=== VIREÁX BASELINE RESULT ===")
print(json.dumps(report, indent=2, default=str))
print("\nReport saved to:", REPORT)

if report["status"] != "PASS":
    print(
        "\nBaseline has NOT been confirmed as passing. "
        "Inspect the error before attempting optimisation."
    )


Running official dfthcsos baseline verifier...
Original input files will not be modified.
This may take approximately 25 seconds, according to the guide.

=== VIREÁX BASELINE RESULT ===
{
  "task": "dfthcsos",
  "timestamp_utc": "2026-10-26T06:09:11.664282+00:00",
  "baseline_directory": "/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission/dfthcsos",
  "baseline_npz_sha256": "14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69",
  "verifier": "/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/verifiers/dfthcsos/verify.py",
  "status": "PASS",
  "dependencies": {
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "kickmix_import": "SUCCESS"
  },
  "result": {
    "status": "PASS",
    "score": 3355599588.3155375,
    "lambda_eff": 49.301369250574396,
    "toffoli_count": 93237.0,
    "num_qubits": 730,
    "brot": 12,
    "batch_size": 8,
    "num_circuits": 10,
    "reaction_depth": 110,
    "num_ops": 2060743,
  